# Лабораторная работа VI. Классификация отзывов Кинопоиска трансформерами

**Датасет:** [Kinopoisk Movie Reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews), тот же набор отзывов, который использовался в лабораторных IV и V.

Цель работы — сравнить три transformer-подхода к классификации тональности: эмбеддинги + классический классификатор, fine-tuning через `Trainer` и смешанное сравнение одной модели в обоих режимах.


## Оглавление

1. [Подготовка окружения](#Подготовка-окружения)
2. [Загрузка данных](#Загрузка-данных)
3. [Первичный анализ данных](#Первичный-анализ-данных)
4. [Предобработка и токенизация](#Предобработка-и-токенизация)
5. [Классификация трансформерами](#Классификация-трансформерами)
6. [Анализ ошибок](#Анализ-ошибок)
7. [Тест на новых данных](#Тест-на-новых-данных)
8. [Выводы](#Выводы)


## Подготовка окружения

Первый блок фиксирует воспроизводимость, подключает библиотеки для анализа данных, визуализации, Hugging Face `transformers` / `datasets` и обучения моделей. Если пакет отсутствует, ошибка будет явно указывать, какую зависимость нужно установить.


### 1.1 Импорты и глобальные настройки

Здесь задаются базовые константы, метки классов, директории кэшей и ограничения учебного запуска. `MAX_PER_CLASS_FOR_MODELLING` ограничивает число объектов на класс только для transformer-экспериментов, потому что fine-tuning на полном наборе из 131k отзывов может занимать часы; EDA при этом считается по всему датасету.


In [ ]:
from __future__ import annotations

import inspect
import json
import math
import os
import random
import re
import shutil
import tempfile
import time
import warnings
from collections import Counter
from pathlib import Path
from typing import Any

MPLCONFIGDIR = Path(tempfile.gettempdir()) / "ml_lab_matplotlib_cache"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from wordcloud import WordCloud
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

SEED = 42
TEXT_COL = "review"
TARGET_COL = "sentiment"
LABEL_ORDER = ["negative", "neutral", "positive"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_ORDER)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
NUM_LABELS = len(LABEL_ORDER)

# Учебные ограничения. Для полного запуска поставьте MAX_PER_CLASS_FOR_MODELLING = None.
MAX_PER_CLASS_FOR_MODELLING: int | None = 1_500
TRANSFORMER_MAX_LENGTH_LIMIT = 512
EMBEDDING_BATCH_SIZE = 32
FINE_TUNE_BATCH_SIZE = 8
FINE_TUNE_EPOCHS = 3
FINE_TUNE_LEARNING_RATE = 2e-5

BASE_TRANSFORMER_MODEL = "cointegrated/rubert-tiny2"
EMBEDDING_MODEL_CONFIGS = [
    {"model_id": "cointegrated/rubert-tiny2", "short_name": "ruBERT-tiny2"},
    {"model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "short_name": "MiniLM-multilingual"},
]
FINE_TUNE_MODEL_CONFIGS = [
    {"model_id": "cointegrated/rubert-tiny2", "short_name": "ruBERT-tiny2"},
]
MIXED_MODEL_ID = "cointegrated/rubert-tiny2"


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Imports OK")


### 1.2 Выбор устройства

Transformer-модели используют CUDA, MPS или CPU. Режим `auto` выбирает лучший доступный вариант, но его можно заменить на `cpu`, `cuda` или `mps` для воспроизводимого запуска в конкретном окружении.


In [ ]:
DEVICE_MODE = "auto"  # варианты: "auto", "cpu", "cuda", "mps"


def get_torch_device_status() -> dict[str, object]:
    return {
        "torch_version": getattr(torch, "__version__", "unknown"),
        "cuda_available": bool(torch.cuda.is_available()),
        "mps_available": bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()),
    }


def resolve_training_device(mode: str = DEVICE_MODE) -> torch.device:
    normalized = str(mode).strip().lower()
    allowed = {"auto", "cpu", "cuda", "mps"}
    if normalized not in allowed:
        raise ValueError(f"DEVICE_MODE должен быть одним из {sorted(allowed)}, получено: {mode!r}")

    status = get_torch_device_status()
    if normalized == "cuda":
        if not status["cuda_available"]:
            raise RuntimeError("Выбран DEVICE_MODE='cuda', но CUDA недоступна.")
        return torch.device("cuda")
    if normalized == "mps":
        if not status["mps_available"]:
            raise RuntimeError("Выбран DEVICE_MODE='mps', но MPS недоступен.")
        return torch.device("mps")
    if normalized == "cpu":
        return torch.device("cpu")
    if status["cuda_available"]:
        return torch.device("cuda")
    if status["mps_available"]:
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = resolve_training_device()
USE_FP16 = DEVICE.type == "cuda"
print("Requested device mode:", DEVICE_MODE)
print("Torch status:", get_torch_device_status())
print("Training device:", DEVICE)
print("FP16 enabled:", USE_FP16)


### 1.3 Функции визуализации и метрик

Как и в предыдущих лабораторных, визуализации вынесены в отдельные функции: это оставляет основные разделы короткими и читабельными.


In [ ]:
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_COLORS = px.colors.qualitative.Set2

BASIC_SW = {
    "и", "в", "на", "что", "это", "как", "с", "из", "а", "но", "по", "за", "то", "так", "же",
    "был", "была", "были", "быть", "всё", "все", "он", "она", "они", "или", "к", "у", "от",
    "для", "его", "её", "их", "ещё", "уже", "о", "об", "при", "до", "со", "чем", "во", "бы",
    "ли", "тут", "тоже", "этот", "этого", "свой", "своя", "свое", "который", "которая", "которые",
}


def show_plotly_figure(fig, title: str | None = None, *, height: int = 480):
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=title,
        title_x=0.02,
        height=height,
        margin=dict(l=40, r=30, t=75 if title else 45, b=40),
        legend_title_text="Класс",
        font=dict(size=12),
    )
    fig.show()


def make_class_color_map(labels) -> dict[str, str]:
    labels = [str(label) for label in labels]
    return {label: PLOTLY_COLORS[i % len(PLOTLY_COLORS)] for i, label in enumerate(labels)}


def plot_class_counts_and_lengths(data: pd.DataFrame):
    counts = data[TARGET_COL].value_counts().reindex(LABEL_ORDER)
    color_map = make_class_color_map(counts.index)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Распределение классов", "Распределение длин отзывов"),
    )
    fig.add_trace(
        go.Bar(
            x=counts.index,
            y=counts.values,
            text=[f"{value:,}" for value in counts.values],
            textposition="outside",
            marker_color=[color_map[str(label)] for label in counts.index],
            showlegend=False,
        ),
        row=1,
        col=1,
    )

    for label in LABEL_ORDER:
        subset = data.loc[data[TARGET_COL] == label, "word_count"]
        fig.add_trace(
            go.Histogram(
                x=subset,
                nbinsx=70,
                opacity=0.55,
                marker_color=color_map[str(label)],
                name=str(label),
            ),
            row=1,
            col=2,
        )

    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text="Класс", row=1, col=1)
    fig.update_yaxes(title_text="Количество отзывов", row=1, col=1)
    fig.update_xaxes(title_text="Количество слов", range=[0, 700], row=1, col=2)
    fig.update_yaxes(title_text="Частота", row=1, col=2)
    return show_plotly_figure(fig, "Классы и длины отзывов", height=500)


def plot_wordclouds_by_class(data: pd.DataFrame, *, max_words: int = 150):
    color_map = make_class_color_map(LABEL_ORDER)
    cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "Greys"]
    fig = make_subplots(
        rows=1,
        cols=len(LABEL_ORDER),
        subplot_titles=[f"Облако слов — {label}" for label in LABEL_ORDER],
    )

    for col_idx, label in enumerate(LABEL_ORDER, start=1):
        texts = data.loc[data[TARGET_COL] == label, TEXT_COL].astype(str)
        words = [
            word.lower()
            for text in texts
            for word in re.findall(r"[а-яёА-ЯЁ]+", text)
            if len(word) > 2 and word.lower() not in BASIC_SW
        ]
        text_for_cloud = " ".join(words)
        if not text_for_cloud:
            continue

        wc = WordCloud(
            width=700,
            height=420,
            background_color="white",
            colormap=cmaps[(col_idx - 1) % len(cmaps)],
            max_words=max_words,
            collocations=False,
        ).generate(text_for_cloud)
        fig.add_trace(go.Image(z=wc.to_array(), hoverinfo="skip", name=label), row=1, col=col_idx)
        fig.update_xaxes(visible=False, row=1, col=col_idx)
        fig.update_yaxes(visible=False, row=1, col=col_idx)

    return show_plotly_figure(
        fig,
        "Облака слов по классам",
        height=500,
    )


def get_top_words(texts, *, n: int = 20, stopwords: set[str] | None = None):
    stopwords = BASIC_SW if stopwords is None else stopwords
    counter = Counter()
    for text in texts.astype(str):
        words = [
            word.lower()
            for word in re.findall(r"[а-яёА-ЯЁ]+", text)
            if len(word) > 2 and word.lower() not in stopwords
        ]
        counter.update(words)
    return counter.most_common(n)


def plot_top_words_by_class(data: pd.DataFrame, *, n: int = 18):
    color_map = make_class_color_map(LABEL_ORDER)
    fig = make_subplots(
        rows=1,
        cols=len(LABEL_ORDER),
        subplot_titles=[f"Топ-{n} слов — {label}" for label in LABEL_ORDER],
    )

    for col_idx, label in enumerate(LABEL_ORDER, start=1):
        top = get_top_words(data.loc[data[TARGET_COL] == label, TEXT_COL], n=n)
        words, freqs = zip(*top)
        fig.add_trace(
            go.Bar(
                x=list(freqs)[::-1],
                y=list(words)[::-1],
                orientation="h",
                marker_color=color_map[label],
                showlegend=False,
                name=label,
            ),
            row=1,
            col=col_idx,
        )
        fig.update_xaxes(title_text="Частота", row=1, col=col_idx)

    return show_plotly_figure(fig, f"Топ-{n} слов по классам", height=560)


def plot_confusion_matrix_plotly(y_true, y_pred, labels=LABEL_ORDER, *, title: str = "Матрица ошибок"):
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    matrix_df = pd.DataFrame(matrix, index=labels, columns=labels)
    fig = px.imshow(
        matrix_df,
        text_auto=True,
        color_continuous_scale="Blues",
        aspect="auto",
        labels=dict(x="Предсказано", y="Истинный класс", color="Количество"),
    )
    fig.update_traces(hovertemplate="Истинный класс: %{y}<br>Предсказано: %{x}<br>Количество: %{z}<extra></extra>")
    return show_plotly_figure(fig, title, height=520)


def compute_cls_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def shorten_text(text: str, max_chars: int = 420) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text if len(text) <= max_chars else text[: max_chars - 3] + "..."


## Загрузка данных

В лабораторных IV и V использовался Kaggle-набор с папками `pos`, `neg`, `neu`. Блок ниже сначала ищет уже подготовленный CSV или raw-папки в `LAB_VI`, `LAB_V` и `LAB_IV`, а если их нет — пытается скачать датасет через `kagglehub` и сохранить prepared-кэш в `LAB_VI/data/dataset`.


### 2.1 Пути, поиск raw-датасета и prepared-кэш

Функции полностью повторяют идею предыдущих ноутбуков: один `.txt`-файл становится одной строкой таблицы, имя папки превращается в метку тональности, а собранная таблица сохраняется для следующих запусков.


In [ ]:
PREPARED_DATA_FILENAME = "kinopoisk_reviews_prepared.csv"
KAGGLE_DATASET = "mikhailklemin/kinopoisks-movies-reviews"
LABEL_DIR_MAP = {"neg": "negative", "neu": "neutral", "pos": "positive"}


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if all((base / lab).exists() for lab in ["LAB_IV", "LAB_V", "LAB_VI"]):
            return base
    raise FileNotFoundError("Не найден корень проекта с LAB_IV, LAB_V и LAB_VI")


PROJECT_ROOT = find_project_root()
LAB_VI_DIR = PROJECT_ROOT / "LAB_VI"
DATA_DIR = LAB_VI_DIR / "data"
DATASET_DIR = DATA_DIR / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_DIR / PREPARED_DATA_FILENAME

DATASET_SEARCH_DIRS = [
    PROJECT_ROOT / "LAB_VI" / "data" / "dataset",
    PROJECT_ROOT / "LAB_V" / "data" / "dataset",
    PROJECT_ROOT / "LAB_IV" / "data" / "dataset",
]


def count_txt_files(root: Path) -> int:
    return sum(1 for _ in root.rglob("*.txt")) if root.exists() else 0


def get_review_dir_counts(root: Path) -> dict[str, int]:
    return {label_dir: count_txt_files(root / label_dir) for label_dir in LABEL_DIR_MAP}


def has_review_dirs(root: Path) -> bool:
    return all((root / label_dir).is_dir() for label_dir in LABEL_DIR_MAP)


def find_review_dirs_root(root: Path) -> Path | None:
    root = Path(root)
    if has_review_dirs(root):
        return root
    if root.exists():
        for candidate in root.rglob("*"):
            if candidate.is_dir() and has_review_dirs(candidate):
                return candidate
    return None


def find_existing_prepared_table() -> Path | None:
    for directory in DATASET_SEARCH_DIRS:
        candidate = directory / PREPARED_DATA_FILENAME
        if candidate.exists() and candidate.stat().st_size > 0:
            return candidate
    return None


def copy_review_dirs(source_root: Path, target_root: Path) -> None:
    source_root = source_root.resolve()
    target_root = target_root.resolve()
    if source_root == target_root:
        return
    target_root.mkdir(parents=True, exist_ok=True)
    for label_dir in LABEL_DIR_MAP:
        src = source_root / label_dir
        dst = target_root / label_dir
        dst.mkdir(parents=True, exist_ok=True)
        for src_file in src.rglob("*.txt"):
            rel_path = src_file.relative_to(src)
            dst_file = dst / rel_path
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst_file)


def download_kaggle_dataset() -> Path:
    try:
        import kagglehub
    except ImportError as exc:
        raise RuntimeError(
            "Папки pos/neg/neu не найдены и не установлен kagglehub. "
            "Положите датасет в LAB_VI/data/dataset или установите kagglehub."
        ) from exc

    return Path(kagglehub.dataset_download(KAGGLE_DATASET))


def ensure_raw_dataset() -> Path:
    for directory in DATASET_SEARCH_DIRS:
        found = find_review_dirs_root(directory)
        if found is not None:
            copy_review_dirs(found, DATASET_DIR)
            return DATASET_DIR

    downloaded_dir = download_kaggle_dataset()
    downloaded_root = find_review_dirs_root(downloaded_dir)
    if downloaded_root is None:
        raise FileNotFoundError(
            "В скачанном Kaggle-датасете не найдены папки pos/neg/neu. "
            f"Проверенный путь: {downloaded_dir.resolve()}"
        )
    copy_review_dirs(downloaded_root, DATASET_DIR)
    return DATASET_DIR


### 2.2 Сборка таблицы отзывов

Каждый отзыв читается с несколькими запасными кодировками. Из имени файла дополнительно извлекаются `movie_id` и `review_id`, чтобы сохранить совместимость со структурой лабораторных IV и V.


In [ ]:
def read_review_text(path: Path) -> str:
    for encoding in ("utf-8-sig", "utf-8", "cp1251"):
        try:
            return path.read_text(encoding=encoding).strip()
        except UnicodeDecodeError:
            continue
    return path.read_text(encoding="utf-8", errors="replace").strip()


def parse_review_ids(path: Path) -> tuple[int | None, int | None]:
    parts = path.stem.split("-", maxsplit=1)
    if len(parts) != 2:
        return None, None
    try:
        return int(parts[0]), int(parts[1])
    except ValueError:
        return None, None


def build_reviews_dataframe(raw_root: Path) -> pd.DataFrame:
    records = []
    total_files = sum(get_review_dir_counts(raw_root).values())
    progress = tqdm(total=total_files, desc="Reading review txt files", dynamic_ncols=True, leave=False)

    with progress:
        for label_dir, sentiment in LABEL_DIR_MAP.items():
            for txt_path in sorted((raw_root / label_dir).rglob("*.txt")):
                movie_id, review_id = parse_review_ids(txt_path)
                records.append({
                    "review": read_review_text(txt_path),
                    "sentiment": sentiment,
                    "source_label_dir": label_dir,
                    "source_file": str(txt_path.relative_to(raw_root)),
                    "movie_id": movie_id,
                    "review_id": review_id,
                })
                progress.update(1)

    if not records:
        raise ValueError(f"В {raw_root.resolve()} не найдено ни одного .txt-отзыва")

    result = pd.DataFrame.from_records(records)
    result["movie_id"] = result["movie_id"].astype("Int64")
    result["review_id"] = result["review_id"].astype("Int64")
    return result


def load_or_build_reviews_dataframe() -> tuple[pd.DataFrame, Path | None, dict[str, int]]:
    existing_table = find_existing_prepared_table()
    if existing_table is not None:
        df_loaded = pd.read_csv(existing_table)
        if existing_table != DATA_PATH:
            df_loaded.to_csv(DATA_PATH, index=False)
        print("Loaded prepared table:", existing_table.resolve())
        return df_loaded, None, {}

    raw_dataset_root = ensure_raw_dataset()
    raw_counts = get_review_dir_counts(raw_dataset_root)
    raw_total = sum(raw_counts.values())
    if raw_total == 0:
        raise ValueError(f"Папки pos/neg/neu найдены в {raw_dataset_root.resolve()}, но .txt-файлов в них нет")

    df_built = build_reviews_dataframe(raw_dataset_root)
    df_built.to_csv(DATA_PATH, index=False)
    print("Prepared table saved:", DATA_PATH.resolve())
    return df_built, raw_dataset_root, raw_counts


df, raw_dataset_root, raw_counts = load_or_build_reviews_dataframe()
print("Project root:", PROJECT_ROOT)
print("Prepared table:", DATA_PATH.resolve())
print("Raw review counts:", raw_counts if raw_counts else "loaded from prepared CSV")
print("Prepared shape:", df.shape)


### 2.3 Быстрый осмотр и нормализация меток

Проверяем форму таблицы, первые строки и приводим метки к трём ожидаемым классам. Это защищает дальнейшие шаги от случайных отличий в именах колонок или меток.


In [ ]:
print("Columns:", df.columns.tolist())
display(df.head())
df.info()

required_columns = {TEXT_COL, TARGET_COL}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"В подготовленной таблице нет обязательных колонок: {sorted(missing_columns)}")

metadata_columns = [
    column
    for column in ["source_label_dir", "source_file", "movie_id", "review_id"]
    if column in df.columns
]

df = df[[TEXT_COL, TARGET_COL, *metadata_columns]].copy()
label_map = {
    "1": "positive", "0": "negative", "-1": "negative",
    "good": "positive", "bad": "negative",
    "pos": "positive", "neg": "negative", "neu": "neutral",
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "позитивный": "positive", "негативный": "negative", "нейтральный": "neutral",
}

df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.lower().replace(label_map)
df.dropna(subset=[TEXT_COL, TARGET_COL], inplace=True)
df[TEXT_COL] = df[TEXT_COL].astype(str)
df = df[df[TEXT_COL].str.strip().astype(bool)].reset_index(drop=True)

df = df[df[TARGET_COL].isin(LABEL_ORDER)].reset_index(drop=True)
print("Shape after cleaning:", df.shape)
print(df[TARGET_COL].value_counts().reindex(LABEL_ORDER))


## Первичный анализ данных

Подробный EDA уже был сделан в `LAB_IV/solution/lab-4_refactor.ipynb` и `LAB_V/solution/lab-5.ipynb`, поэтому здесь оставлены только результаты, которые нужны для transformer-пайплайна: распределение классов, вид таблицы, длины текстов, уникальные слова, топ-слова и выбор `max_length`.


### 3.1 Распределение классов и длина отзывов

Считаем количество слов через `str.split()`, количество уникальных слов в отзыве и строим два самых полезных графика: баланс классов и распределение длин.


In [ ]:
df["word_count"] = df[TEXT_COL].str.split().str.len()
df["unique_word_count"] = df[TEXT_COL].str.split().map(lambda words: len(Counter(word.lower() for word in words)))

class_counts = df[TARGET_COL].value_counts().reindex(LABEL_ORDER)
length_stats = df.groupby(TARGET_COL)["word_count"].agg(
    count="count",
    mean="mean",
    median="median",
    p95=lambda x: x.quantile(0.95),
    max="max",
).round(1).reindex(LABEL_ORDER)
unique_stats = df.groupby(TARGET_COL)["unique_word_count"].agg(mean="mean", median="median").round(1).reindex(LABEL_ORDER)

print("Распределение классов:")
display(class_counts.to_frame("count"))
print("\nСтатистика длины отзывов в словах:")
display(length_stats)
print("\nУникальные слова в отдельных отзывах:")
display(unique_stats)

plot_class_counts_and_lengths(df)


### 3.2 Топ-слова по классам

Вместо повторения всех облаков слов из предыдущих лабораторных оставляем одну количественную визуализацию: топ частотных слов отдельно для каждого класса после грубой фильтрации служебных слов.


In [ ]:
plot_wordclouds_by_class(df, max_words=150)
plot_top_words_by_class(df, n=18)

common_words = Counter()
for text in tqdm(df[TEXT_COL].astype(str), desc="Counting vocabulary", dynamic_ncols=True):
    common_words.update(word.lower() for word in re.findall(r"[а-яёА-ЯЁ]+", text))

print(f"Всего токенов:      {sum(common_words.values()):>12,}")
print(f"Уникальных токенов: {len(common_words):>12,}")
print("Топ-20 слов всего:")
display(pd.DataFrame(common_words.most_common(20), columns=["word", "count"]))


### 3.3 Выбор `max_length`

По заданию `max_length` выбирается на основе 95-го перцентиля длины отзывов. Для BERT-подобных моделей дополнительно учитывается архитектурный предел 512 токенов: если 95-й перцентиль в словах выше этого лимита, берём максимально допустимые 512 и явно фиксируем, что длинные отзывы будут обрезаться.


In [ ]:
word_length_p95 = int(math.ceil(df["word_count"].quantile(0.95)))
TRANSFORMER_MAX_LENGTH = min(TRANSFORMER_MAX_LENGTH_LIMIT, max(64, word_length_p95))
truncation_share = float((df["word_count"] > TRANSFORMER_MAX_LENGTH).mean())

print(f"95-й перцентиль длины отзыва: {word_length_p95} слов")
print(f"Выбранный TRANSFORMER_MAX_LENGTH: {TRANSFORMER_MAX_LENGTH}")
print(f"Доля отзывов длиннее выбранного лимита: {truncation_share:.1%}")


## Предобработка и токенизация

Для трансформеров не нужна ручная лемматизация из предыдущих лабораторных: модель получает исходный русский текст, а всю модельно-специфичную обработку делает `AutoTokenizer`. Здесь готовим стратифицированные train/validation/test split и проверяем токенизацию.


### 4.1 Стратифицированный сэмпл для обучения

Полный датасет остаётся в `df`, а transformer-эксперименты запускаются на сбалансированном по классам учебном сэмпле. Это делает ноутбук исполнимым на CPU/MPS и сохраняет возможность перейти на полный набор одной настройкой.


In [ ]:
def make_stratified_sample(data: pd.DataFrame, per_class: int | None, seed: int = SEED) -> pd.DataFrame:
    if per_class is None:
        return data.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    parts = []
    for label, group in data.groupby(TARGET_COL, sort=False):
        n = min(per_class, len(group))
        parts.append(group.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)


model_df = make_stratified_sample(df, MAX_PER_CLASS_FOR_MODELLING)
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=SEED,
    stratify=model_df[TARGET_COL],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df[TARGET_COL],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Model sample shape:", model_df.shape)
print("Train / val / test:", len(train_df), len(val_df), len(test_df))
print("Train class balance:")
display(train_df[TARGET_COL].value_counts().reindex(LABEL_ORDER).to_frame("count"))


### 4.2 Токенизация базовой модели

Показываем, как `AutoTokenizer` превращает текст в `input_ids` и `attention_mask`. Этот же `max_length` затем используется и для эмбеддингов, и для fine-tuning.


In [ ]:
base_tokenizer = AutoTokenizer.from_pretrained(BASE_TRANSFORMER_MODEL)
preview_texts = train_df[TEXT_COL].head(3).tolist()
preview_tokens = base_tokenizer(
    preview_texts,
    truncation=True,
    padding=True,
    max_length=TRANSFORMER_MAX_LENGTH,
)

print("Tokenizer:", BASE_TRANSFORMER_MODEL)
print("Tokenizer model_max_length:", base_tokenizer.model_max_length)
print("Keys:", preview_tokens.keys())
print("input_ids shape:", np.array(preview_tokens["input_ids"]).shape)
print("Первый текст:", shorten_text(preview_texts[0], 260))
print("Первые 30 token ids:", preview_tokens["input_ids"][0][:30])


### 4.3 Hugging Face Dataset

`Trainer` ожидает `datasets.Dataset`, поэтому добавляем числовую колонку `label` и собираем `DatasetDict` для трёх split. Тексты остаются исходными, без ручной очистки.


In [ ]:
def add_label_ids(data: pd.DataFrame) -> pd.DataFrame:
    result = data[[TEXT_COL, TARGET_COL]].copy()
    result["label"] = result[TARGET_COL].map(LABEL_TO_ID).astype(int)
    return result


hf_datasets = DatasetDict({
    "train": Dataset.from_pandas(add_label_ids(train_df), preserve_index=False),
    "validation": Dataset.from_pandas(add_label_ids(val_df), preserve_index=False),
    "test": Dataset.from_pandas(add_label_ids(test_df), preserve_index=False),
})

print(hf_datasets)
display(hf_datasets["train"].to_pandas().head())


## Классификация трансформерами

Основной раздел использует все три подхода из задания:

1. `AutoModel` как генератор sentence embeddings + `LogisticRegression`.
2. `AutoModelForSequenceClassification` + `Trainer` для fine-tuning.
3. Смешанное сравнение одной и той же модели `ruBERT-tiny2` в двух режимах: эмбеддинги против fine-tuning, включая F1 и время.


### 5.0 Контроль тяжелых ячеек

Дальше начинаются ячейки, которые скачивают модели Hugging Face и запускают обучение или построение эмбеддингов. В этой рабочей копии они оставлены без outputs и не выполнялись; запускать их нужно на машине, где есть датасет, доступ к моделям и достаточно времени.


### 5.1 Общие функции для transformer-экспериментов

Метрики, pooling и служебные функции одинаковы для всех подходов. Mean pooling усредняет скрытые состояния только по реальным токенам, игнорируя padding через `attention_mask`.


In [ ]:
def labels_to_ids(labels: pd.Series | list[str]) -> np.ndarray:
    return np.array([LABEL_TO_ID[str(label)] for label in labels], dtype=np.int64)


def ids_to_labels(label_ids: np.ndarray | list[int]) -> np.ndarray:
    return np.array([ID_TO_LABEL[int(label_id)] for label_id in label_ids])


def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    true_labels = ids_to_labels(y_true)
    pred_labels = ids_to_labels(y_pred)
    return {
        "test_accuracy": accuracy_score(true_labels, pred_labels),
        "test_macro_f1": f1_score(true_labels, pred_labels, average="macro"),
    }


experiment_rows: list[dict[str, Any]] = []
model_outputs: dict[str, dict[str, Any]] = {}


### 5.2 Подход 1 — sentence embeddings + LogisticRegression

Здесь трансформер не дообучается. Он один раз превращает отзывы в dense-векторы, после чего поверх эмбеддингов обучается быстрый линейный классификатор с балансировкой классов.


In [ ]:
@torch.inference_mode()
def encode_texts_to_embeddings(
    texts: list[str],
    tokenizer,
    encoder,
    *,
    batch_size: int = EMBEDDING_BATCH_SIZE,
    max_length: int = TRANSFORMER_MAX_LENGTH,
) -> np.ndarray:
    encoder.eval()
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Encoding embeddings", dynamic_ncols=True):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        outputs = encoder(**encoded)
        pooled = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])
        vectors.append(F.normalize(pooled, p=2, dim=1).cpu().numpy())
    return np.vstack(vectors)


def run_embedding_classifier(config: dict[str, str]) -> dict[str, Any]:
    model_id = config["model_id"]
    short_name = config["short_name"]
    print(f"\n=== Embeddings: {short_name} ({model_id}) ===")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    encoder = AutoModel.from_pretrained(model_id).to(DEVICE)

    t0 = time.perf_counter()
    X_train_emb = encode_texts_to_embeddings(train_df[TEXT_COL].tolist(), tokenizer, encoder)
    X_val_emb = encode_texts_to_embeddings(val_df[TEXT_COL].tolist(), tokenizer, encoder)
    train_embedding_seconds = time.perf_counter() - t0

    classifier = LogisticRegression(
        max_iter=1_000,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    t0 = time.perf_counter()
    classifier.fit(X_train_emb, train_df[TARGET_COL])
    classifier_seconds = time.perf_counter() - t0

    val_pred = classifier.predict(X_val_emb)
    val_metrics = compute_cls_metrics(val_df[TARGET_COL], val_pred)

    t0 = time.perf_counter()
    X_test_emb = encode_texts_to_embeddings(test_df[TEXT_COL].tolist(), tokenizer, encoder)
    test_proba = classifier.predict_proba(X_test_emb)
    test_pred = classifier.classes_[np.argmax(test_proba, axis=1)]
    predict_seconds = time.perf_counter() - t0
    test_metrics = compute_cls_metrics(test_df[TARGET_COL], test_pred)

    key = f"embeddings::{short_name}"
    row = {
        "key": key,
        "approach": "embeddings",
        "model_id": model_id,
        "model": short_name,
        "classifier": "LogisticRegression",
        "validation_accuracy": val_metrics["accuracy"],
        "validation_macro_f1": val_metrics["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "train_seconds": train_embedding_seconds + classifier_seconds,
        "predict_seconds": predict_seconds,
    }
    experiment_rows.append(row)
    model_outputs[key] = {
        "approach": "embeddings",
        "model_id": model_id,
        "model": short_name,
        "tokenizer": tokenizer,
        "encoder": encoder,
        "classifier": classifier,
        "test_predictions": np.array(test_pred),
        "test_probabilities": test_proba,
        "class_order": classifier.classes_,
    }
    return row


embedding_rows = [run_embedding_classifier(config) for config in EMBEDDING_MODEL_CONFIGS]
pd.DataFrame(embedding_rows).sort_values("test_macro_f1", ascending=False)


### 5.3 Подход 2 — fine-tuning через Trainer

Во втором подходе классификационная голова обучается вместе с весами трансформера. Для компактного учебного запуска используется `ruBERT-tiny2`, но конфигурация вынесена в список, поэтому второй fine-tuning-моделью можно добавить, например, `DeepPavlov/rubert-base-cased`.


In [ ]:
def tokenize_dataset(dataset: DatasetDict, tokenizer) -> DatasetDict:
    def tokenize_batch(batch):
        return tokenizer(
            batch[TEXT_COL],
            truncation=True,
            max_length=TRANSFORMER_MAX_LENGTH,
        )

    tokenized = dataset.map(tokenize_batch, batched=True)
    return tokenized.remove_columns([TEXT_COL, TARGET_COL])


def compute_trainer_metrics(eval_pred) -> dict[str, float]:
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    labels = eval_pred.label_ids
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


def make_training_arguments(output_dir: Path) -> TrainingArguments:
    common_kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=FINE_TUNE_LEARNING_RATE,
        per_device_train_batch_size=FINE_TUNE_BATCH_SIZE,
        per_device_eval_batch_size=FINE_TUNE_BATCH_SIZE * 2,
        num_train_epochs=FINE_TUNE_EPOCHS,
        weight_decay=0.01,
        logging_steps=50,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        fp16=USE_FP16,
    )

    signature = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in signature.parameters:
        common_kwargs.update({"eval_strategy": "epoch", "save_strategy": "epoch"})
    else:
        common_kwargs.update({"evaluation_strategy": "epoch", "save_strategy": "epoch"})

    if "use_cpu" in signature.parameters:
        common_kwargs["use_cpu"] = DEVICE.type == "cpu"

    return TrainingArguments(**common_kwargs)


def run_fine_tuning(config: dict[str, str]) -> dict[str, Any]:
    model_id = config["model_id"]
    short_name = config["short_name"]
    print(f"\n=== Fine-tuning: {short_name} ({model_id}) ===")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenized_datasets = tokenize_dataset(hf_datasets, tokenizer)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=NUM_LABELS,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
    )

    output_dir = DATASET_DIR / "transformer_runs" / re.sub(r"[^a-zA-Z0-9_.-]+", "_", short_name)
    args = make_training_arguments(output_dir)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_trainer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.perf_counter()
    train_output = trainer.train()
    train_seconds = time.perf_counter() - t0

    val_metrics = trainer.evaluate(tokenized_datasets["validation"])

    t0 = time.perf_counter()
    test_output = trainer.predict(tokenized_datasets["test"])
    predict_seconds = time.perf_counter() - t0
    logits = test_output.predictions[0] if isinstance(test_output.predictions, tuple) else test_output.predictions
    test_probabilities = softmax_np(logits)
    test_pred_ids = np.argmax(logits, axis=1)
    test_pred = ids_to_labels(test_pred_ids)
    test_metrics = compute_cls_metrics(test_df[TARGET_COL], test_pred)

    key = f"fine_tune::{short_name}"
    row = {
        "key": key,
        "approach": "fine_tuning",
        "model_id": model_id,
        "model": short_name,
        "classifier": "AutoModelForSequenceClassification",
        "validation_accuracy": val_metrics.get("eval_accuracy", np.nan),
        "validation_macro_f1": val_metrics.get("eval_macro_f1", np.nan),
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "train_seconds": train_seconds,
        "predict_seconds": predict_seconds,
        "train_loss": getattr(train_output, "training_loss", np.nan),
    }
    experiment_rows.append(row)
    model_outputs[key] = {
        "approach": "fine_tuning",
        "model_id": model_id,
        "model": short_name,
        "tokenizer": tokenizer,
        "trainer": trainer,
        "tokenized_datasets": tokenized_datasets,
        "test_predictions": np.array(test_pred),
        "test_probabilities": test_probabilities,
        "class_order": np.array(LABEL_ORDER),
    }
    return row


fine_tune_rows = [run_fine_tuning(config) for config in FINE_TUNE_MODEL_CONFIGS]
pd.DataFrame(fine_tune_rows).sort_values("test_macro_f1", ascending=False)


### 5.4 Подход 3 — смешанное сравнение одной модели

Третий подход — это не новая архитектура, а честное сравнение одного backbone в двух режимах. Для `ruBERT-tiny2` сопоставляем F1, accuracy и время: насколько fine-tuning выигрывает или проигрывает у эмбеддингового режима.


In [ ]:
results_df = pd.DataFrame(experiment_rows).sort_values("test_macro_f1", ascending=False).reset_index(drop=True)
display(results_df[[
    "approach", "model", "classifier", "validation_macro_f1", "test_macro_f1", "test_accuracy", "train_seconds", "predict_seconds"
]])

mixed_comparison = results_df[results_df["model_id"] == MIXED_MODEL_ID].copy()
mixed_comparison = mixed_comparison.sort_values("approach")

if len(mixed_comparison) >= 2:
    best_mixed = mixed_comparison.sort_values("test_macro_f1", ascending=False).iloc[0]
    baseline_mixed = mixed_comparison.sort_values("test_macro_f1", ascending=False).iloc[-1]
    print(f"Лучшая версия {MIXED_MODEL_ID}: {best_mixed['approach']} ({best_mixed['test_macro_f1']:.4f} macro-F1)")
    print(f"Разница с альтернативным режимом: {best_mixed['test_macro_f1'] - baseline_mixed['test_macro_f1']:+.4f} macro-F1")
else:
    print("Для mixed-сравнения нужен один и тот же model_id минимум в двух подходах.")

display(mixed_comparison[["approach", "model", "test_macro_f1", "test_accuracy", "train_seconds", "predict_seconds"]])


### 5.5 Выбор лучшей модели

Лучшая модель выбирается по `test_macro_f1`, потому что классы в исходном датасете сильно несбалансированы. Дальше именно она используется для анализа ошибок и новых текстов.


In [ ]:
if results_df.empty:
    raise RuntimeError("Нет результатов экспериментов. Сначала выполните ячейки подходов 1 и 2.")

best_row = results_df.iloc[0]
best_key = best_row["key"]
best_output = model_outputs[best_key]

print("Best model key:", best_key)
print("Best approach:", best_row["approach"])
print("Best model:", best_row["model"])
print(f"Best test macro-F1: {best_row['test_macro_f1']:.4f}")
print(f"Best test accuracy: {best_row['test_accuracy']:.4f}")


## Анализ ошибок

Для лучшей модели смотрим confusion matrix и конкретные ошибки. Это важнее одной итоговой метрики: в отзывах Кинопоиска особенно часто путаются `neutral` и соседние классы, потому что нейтральные рецензии могут содержать и похвалу, и критику.


### 6.1 Матрица ошибок и classification report

Матрица показывает направления перепутывания классов, а `classification_report` — precision, recall и F1 отдельно для `negative`, `neutral`, `positive`.


In [ ]:
best_predictions = best_output["test_predictions"]
plot_confusion_matrix_plotly(test_df[TARGET_COL], best_predictions, title=f"Матрица ошибок: {best_row['model']} / {best_row['approach']}")

report = classification_report(
    test_df[TARGET_COL],
    best_predictions,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T.round(4)
display(report_df)


### 6.2 Примеры неверных предсказаний

Выводим до пяти ошибок с вероятностями классов. Для содержательного разбора достаточно короткого фрагмента исходного отзыва и пары `истинный класс → предсказанный класс`.


In [ ]:
probabilities = best_output["test_probabilities"]
class_order = list(best_output["class_order"])
confidence = probabilities.max(axis=1)

errors_df = test_df[[TEXT_COL, TARGET_COL]].copy()
errors_df["predicted_label"] = best_predictions
errors_df["confidence"] = confidence
for idx, label in enumerate(class_order):
    errors_df[f"p_{label}"] = probabilities[:, idx]
errors_df["short_review"] = errors_df[TEXT_COL].map(shorten_text)
errors_df = errors_df[errors_df[TARGET_COL] != errors_df["predicted_label"]].copy()

print(f"Ошибок: {len(errors_df)} из {len(test_df)} ({len(errors_df) / len(test_df):.1%})")
error_examples = errors_df.sort_values("confidence", ascending=False).head(5)
display(error_examples[["short_review", TARGET_COL, "predicted_label", "confidence", *[f"p_{label}" for label in class_order]]])


### 6.3 Краткая интерпретация ошибок

Сводим направления ошибок в таблицу. По ней видно, какие пары классов требуют больше данных, другой балансировки или более длинного контекста.


In [ ]:
error_pairs = (
    errors_df.groupby([TARGET_COL, "predicted_label"])
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)
display(error_pairs.head(10))


## Тест на новых данных

Пять коротких отзывов написаны вручную так, чтобы покрыть явно положительную, явно отрицательную и смешанную тональность. Они проходят через тот же tokenizer / embedding / Trainer-пайплайн, что и тестовая выборка.


### 7.1 Новые отзывы и функция инференса

Функция `predict_new_texts` автоматически выбирает нужную ветку: для embedding-подхода строит эмбеддинги и применяет `LogisticRegression`, для fine-tuning использует `Trainer.predict`.


In [ ]:
new_examples = [
    {"text": "Фильм получился живым и смешным, актеры отлично держат темп.", "expected_label": "positive"},
    {"text": "Сюжет разваливается уже к середине, финал совсем не работает.", "expected_label": "negative"},
    {"text": "Картина снята аккуратно, но сильных эмоций после просмотра не осталось.", "expected_label": "neutral"},
    {"text": "Музыка и визуальный стиль великолепны, хотя отдельные сцены затянуты.", "expected_label": "positive"},
    {"text": "Не худший вечерний фильм, но пересматривать его точно не хочется.", "expected_label": "neutral"},
]
new_examples_df = pd.DataFrame(new_examples)


def predict_new_texts(output: dict[str, Any], texts: list[str]) -> pd.DataFrame:
    if output["approach"] == "embeddings":
        embeddings = encode_texts_to_embeddings(texts, output["tokenizer"], output["encoder"])
        probabilities = output["classifier"].predict_proba(embeddings)
        class_order = list(output["classifier"].classes_)
        predictions = output["classifier"].classes_[np.argmax(probabilities, axis=1)]
    elif output["approach"] == "fine_tuning":
        tokenizer = output["tokenizer"]
        dataset = Dataset.from_dict({TEXT_COL: texts, "label": [0] * len(texts)})
        tokenized = dataset.map(
            lambda batch: tokenizer(batch[TEXT_COL], truncation=True, max_length=TRANSFORMER_MAX_LENGTH),
            batched=True,
        ).remove_columns([TEXT_COL])
        pred_output = output["trainer"].predict(tokenized)
        logits = pred_output.predictions[0] if isinstance(pred_output.predictions, tuple) else pred_output.predictions
        probabilities = softmax_np(logits)
        class_order = LABEL_ORDER
        predictions = ids_to_labels(np.argmax(probabilities, axis=1))
    else:
        raise ValueError(f"Неизвестный approach: {output['approach']}")

    result = pd.DataFrame({"text": texts, "predicted_label": predictions})
    result["confidence"] = probabilities.max(axis=1)
    for idx, label in enumerate(class_order):
        result[f"p_{label}"] = probabilities[:, idx]
    return result


new_predictions = predict_new_texts(best_output, new_examples_df["text"].tolist())
new_predictions.insert(1, "expected_label", new_examples_df["expected_label"])
new_predictions["is_expected"] = new_predictions["expected_label"] == new_predictions["predicted_label"]
display(new_predictions)


## Выводы

В этой лабораторной используется тот же датасет Кинопоиска, что и в лабораторных IV и V, но классификация перенесена на transformer-подходы. Первичный анализ оставлен кратким: распределение классов, статистика длины и топ-слова нужны только для выбора `max_length` и интерпретации ошибок.

После выполнения всех ячеек итоговая таблица `results_df` сравнивает три требуемых варианта: эмбеддинги + `LogisticRegression`, fine-tuning через `Trainer` и mixed-сравнение одной модели в двух режимах. Для итогового выбора используется `macro-F1`, потому что исходный набор несбалансирован: положительных отзывов существенно больше, чем нейтральных и отрицательных.

Дальнейшие улучшения: запустить fine-tuning на полном датасете, добавить вторую крупную русскоязычную модель (`DeepPavlov/rubert-base-cased` или аналог), подобрать learning rate / epochs через validation и отдельно исследовать длинные отзывы, которые обрезаются до 512 токенов.
